# Gradient Boosting Classifier: Iris Dataset

## Overview

**Boosting** is an ensemble method that trains models sequentially. Each new model focuses on correcting the errors made by previous models. **Gradient Boosting** specifically uses gradient descent to optimize the loss function, resulting in powerful predictive models.

## Key Concept: How Boosting Works

1. **Sequential Training:** Train models one at a time
2. **Error Focusing:** Each new model focuses on samples misclassified by previous models
3. **Weighted Predictions:** Combine all models' predictions with weighted averaging
4. **Result:** Stronger learning signal, reduced bias and variance

## Bagging vs. Boosting

| Aspect | Bagging | Boosting |
|--------|---------|----------|
| Training | Parallel | Sequential |
| Focus | All samples equally | Misclassified samples |
| Effect | Reduces variance | Reduces bias & variance |
| Parallelizable | Yes | No |

## Learning Objectives

By the end of this notebook, you will understand:
1. How Boosting works and differs from Bagging
2. How to train a Gradient Boosting Classifier
3. How learning_rate and n_estimators affect performance
4. How to evaluate boosting models

## Step 1: Import Libraries

In [ ]:
from matplotlib import pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

## Step 2: Load and Prepare Data

In [ ]:
# Load the Iris dataset
iris_ds = load_iris()
X = iris_ds.data
y = iris_ds.target

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Classes: {iris_ds.target_names}')

## Step 3: Train Single Decision Tree (Baseline)

In [ ]:
# Single Decision Tree (baseline)
dt_single = DecisionTreeClassifier(
    criterion='gini',
    max_depth=3,
    random_state=42
)
dt_single.fit(X_train, y_train)

y_pred_single = dt_single.predict(X_test)
acc_single = accuracy_score(y_test, y_pred_single)

print(f'Single Tree Accuracy: {acc_single:.4f}')

## Step 4: Train Gradient Boosting Classifier

Gradient Boosting trains weak learners (shallow trees) sequentially. Key parameters:
- **n_estimators:** Number of boosting stages (trees) to build
- **learning_rate:** Shrinks contributions of each tree (lower = slower but often better)
- **max_depth:** Depth of individual trees (typically small: 3-8)
- **subsample:** Fraction of samples used for fitting (helps reduce variance)

In [ ]:
# Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(
    n_estimators=100,        # Number of boosting stages
    learning_rate=0.1,       # Shrinks the contribution of each tree
    max_depth=3,             # Depth of individual trees (weak learners)
    subsample=0.8,           # Use 80% of samples for each tree
    random_state=42
)

# Train the boosting ensemble
gb_clf.fit(X_train, y_train)

# Evaluate
y_pred_gb = gb_clf.predict(X_test)
acc_gb = accuracy_score(y_test, y_pred_gb)

print(f'Gradient Boosting Accuracy: {acc_gb:.4f}')
print(f'Improvement over single tree: {(acc_gb - acc_single):.4f}')

## Step 5: Effect of Learning Rate

The learning rate controls how much each new tree contributes to the ensemble. Lower learning rates often generalize better but require more estimators.

In [ ]:
# Test with different learning rates
learning_rates = [0.01, 0.05, 0.1, 0.2]
lr_train_accs = []
lr_test_accs = []

for lr in learning_rates:
    gb = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=lr,
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, gb.predict(X_train))
    test_acc = accuracy_score(y_test, gb.predict(X_test))
    
    lr_train_accs.append(train_acc)
    lr_test_accs.append(test_acc)
    print(f'Learning Rate {lr}: Train = {train_acc:.4f}, Test = {test_acc:.4f}')

## Step 6: Effect of Number of Estimators

In [ ]:
# Test with different numbers of estimators
n_estimators_list = [10, 20, 50, 100, 200]
ne_train_accs = []
ne_test_accs = []

for n_est in n_estimators_list:
    gb = GradientBoostingClassifier(
        n_estimators=n_est,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
    gb.fit(X_train, y_train)
    
    train_acc = accuracy_score(y_train, gb.predict(X_train))
    test_acc = accuracy_score(y_test, gb.predict(X_test))
    
    ne_train_accs.append(train_acc)
    ne_test_accs.append(test_acc)
    print(f'n_estimators={n_est:3d}: Train = {train_acc:.4f}, Test = {test_acc:.4f}')

## Step 7: Visualize Training Progress

In [ ]:
# Plot training and test accuracy vs. number of estimators
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Learning rate effect
axes[0].plot(learning_rates, lr_train_accs, 'o-', linewidth=2, markersize=8, label='Train')
axes[0].plot(learning_rates, lr_test_accs, 's-', linewidth=2, markersize=8, label='Test')
axes[0].set_xlabel('Learning Rate')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Effect of Learning Rate (n_estimators=100)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Number of estimators effect
axes[1].plot(n_estimators_list, ne_train_accs, 'o-', linewidth=2, markersize=8, label='Train')
axes[1].plot(n_estimators_list, ne_test_accs, 's-', linewidth=2, markersize=8, label='Test')
axes[1].set_xlabel('Number of Estimators')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Effect of Number of Estimators (lr=0.1)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 8: Detailed Evaluation

In [ ]:
# Final model evaluation
final_gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
final_gb.fit(X_train, y_train)
y_pred_final = final_gb.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred_final, target_names=iris_ds.target_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_final)
print('\nConfusion Matrix:')
print(cm)

## Key Takeaways

1. **Sequential training:** Boosting trains models one at a time, each focusing on previous errors
2. **Reduced bias:** Better captures complex patterns than single trees
3. **Learning rate trade-off:** Lower learning rates generalize better but need more estimators
4. **Weak learners:** Individual trees are intentionally kept shallow (max_depth typically 3-8)
5. **Not parallelizable:** Sequential nature prevents parallel training (unlike Bagging)
6. **Better performance:** Gradient Boosting often achieves better accuracy than Bagging
7. **Overfitting risk:** Can overfit if n_estimators or learning_rate are too high
8. **Computational cost:** More expensive to train than Bagging (sequential vs. parallel)